In [ ]:
import requests
import json
from typing import Tuple

"""
Queries the WEED STAC API for feature cube items based on the provided spatial and temporal extents.
Returns a boolean indicating the presence of items if the query is successful or None otherwise. The
spatial extent is provided as a bounding box, and the temporal extent must be defined by start and
end date strings.

:param spatial_extent: The bounding box for the geographical area of interest, defined as
    (min_lon, min_lat, max_lon, max_lat).
:param temporal_extent: The time range for the query, given as a tuple of start and end date
    strings in the format "YYYY-MM-DD".
:return: True if one or more items matching the criteria are found, False if no items are found,
    or None if the query does not succeed.
:raises RuntimeError: If a timeout occurs, an error is encountered during the request, or if an
    unexpected response or status code is received.
"""


# search endpoint of the WEED STAC API
search_endpoint = " https://earth-search.aws.element84.com/v1/search"
# create the search string
search_payload = {
    'limit': 20, 
    'collections': ['sentinel-2-pre-c1-l2a'], 
    'filter-lang': 'cql2-json', 
    'bbox': [4.785959977018684, 52.3367728809811, 4.919564507100708, 52.39865946645364], 
    #'datetime': '2019-12-31T23:00:00+00:00/2021-12-31T22:59:59+00:00'
}

# execute the search
try:
    r = requests.post(search_endpoint, json=search_payload, timeout=(3, 5))
except requests.exceptions.Timeout:
    raise RuntimeError("Timeout while searching for feature items")
except requests.exceptions.RequestException as e:
    raise RuntimeError(f"Error while searching: {e}")

# handle response - here we have the rule that modelIDs have to be UNIQUE
if r.status_code == 200:
    search_results = r.json()
    if len(search_results["features"]) >= 1:
        print(True)
    elif len(search_results["features"]) == 0:
        print(False)
elif r.status_code == 400:
    print("Bad Request – validation errors:")
    raise RuntimeError(json.dumps(r.json(), indent=2))
else:
    print(f"Unexpected status {r.status_code}:")
    raise RuntimeError(r.text)

False
